# Do Macro-Financial Signals Improve US Equity Factor Allocation?

*An exploratory study of prediction, allocation and economic states, with UK and German cross-market checks*

## Goal

Rebuild the fixed exploratory study from the bundled raw data. The primary Macro-minus-historical-mean Sharpe increment is approximately 0.055, with a 95% interval crossing zero. State forecasting has conditional RMW evidence in the extended window; the original 88-month window does not reject at 5%. Absolute gains are concentrated in 2022, but that year accounts for about 30% of the paired arithmetic increment. This notebook rebuilds the models, costs, inference and eight figures. Word report generation is disabled; write the report separately. The report/ folder is not required for this notebook.

Keep the entire extracted project together; the notebook calls the inspectable modules in `src/`.

## 1. Locate the complete project / 定位项目

推荐先按照 `userguide.md` 创建独立 Python 3.13 环境，在项目目录运行 `python start_jupyter.py`，再打开本 Notebook。启动器会定位自己的项目目录，不依赖原作者的电脑路径。

从 Anaconda Navigator 启动时，若当前工作目录不同，请在下一格的 `PROJECT_DIRECTORY` 中填写**你自己电脑上**完整解压目录。不要填写 Notebook 文件名。默认 `RUN_FULL_ANALYSIS = True` 会重做全部计算。

启动器提供 **UTSG7394 (project Python)** 临时内核。若通过 Navigator 打开时没有这个内核，请选择已安装本项目依赖的 Python 内核。

In [ ]:
from pathlib import Path
import json
import runpy
import subprocess
import sys
import os

PROJECT_DIRECTORY = os.environ.get("UTSG7394_PROJECT_DIRECTORY", "")  # Optional: complete extracted directory, e.g. r"C:\projects\UTSG7394" # directory in laptop
RUN_FULL_ANALYSIS = True

def complete_project(path):
    return all((path / name).is_file() for name in
               ["run_study.py", "start_jupyter.py", "specification.json", "data/input_manifest.json"])

start = Path.cwd().resolve()
print("当前工作目录 / Working directory:", start)
print("当前 Python / Kernel interpreter:", sys.executable)
if PROJECT_DIRECTORY.strip():
    PROJECT = Path(PROJECT_DIRECTORY).expanduser().resolve()
else:
    PROJECT = next((path for path in [start, *start.parents] if complete_project(path)), None)
    if PROJECT is None:
        # A single immediate child is safe to infer. Multiple copies are ambiguous.
        candidates = [path for path in start.iterdir() if path.is_dir() and complete_project(path)]
        if len(candidates) == 1:
            PROJECT = candidates[0]
        elif len(candidates) > 1:
            raise FileNotFoundError("找到多个项目，请在 PROJECT_DIRECTORY 指定本次完整解压目录。\n" +
                                    "\n".join(str(path) for path in candidates))
if PROJECT is None or not complete_project(PROJECT):
    raise FileNotFoundError(
        "未找到完整项目。请在 PROJECT_DIRECTORY 填写包含 run_study.py、data 和 src 的目录。\n"
        "推荐从该目录运行 python start_jupyter.py 后再打开本 Notebook。\n"
        "Find the extracted project folder; a notebook file alone is insufficient.")
print("项目已找到 / Project located:", PROJECT)
print("Mode:", "Full raw-data reconstruction" if RUN_FULL_ANALYSIS
      else "Validate existing results and rebuild figures")

## 2. Optional dependency installation / 可选安装依赖

已用 `environment.yml` 建好环境时，保持下一格 `INSTALL_DEPENDENCIES = False`。

若后续预检提示缺包，可将其改为 `True` 并**单独运行这一格**；它会安装到当前 Notebook 的 Python 环境，可能升级已有包。推荐使用独立环境，避免影响 Anaconda base。安装结束后选择 **Kernel → Restart Kernel**，把开关改回 `False`，再 **Run → Run All Cells**。本 Notebook 默认运行不会静默安装依赖。安装需要联网；研究计算离线。

In [ ]:
INSTALL_DEPENDENCIES = False  # Explicit opt-in; never installs silently during Run All.
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(PROJECT / "requirements-notebook.txt")], cwd=PROJECT)
    print("安装完成。请重启内核，把 INSTALL_DEPENDENCIES 改回 False 后重新 Run All。")
else:
    print("未执行安装 / Dependency installation skipped.")

## 3. Check the environment / 环境预检

预检显示当前解释器、缺失依赖及与测试版本的差异。`docx` 的安装包名是 `python-docx`。缺包时会给出可复制的当前内核安装命令；实际数值一致性仍由后续计算核对。

In [ ]:
startup = runpy.run_path(str(PROJECT / "start_jupyter.py"))
environment = startup["require_environment"](include_notebook=True)
print("环境可用于开始计算；已测试 Python 为 3.13。")

## Data and fixed methods

The original data files and configuration are checked before computation. Current-vintage macro histories are release-aware, while the industrial-production sensitivity uses historical decision-month vintages. The latter is a partial real-time check. Reference CSVs are numerical comparison targets; they do not supply fitted forecasts or portfolio returns.

In [ ]:
manifest = json.loads((PROJECT / "data/input_manifest.json").read_text(encoding="utf-8"))
configuration = json.loads((PROJECT / "specification.json").read_text(encoding="utf-8"))
print("Original source files:", len(manifest["files"]))
print("Configuration:", "specification.json")
print("Bootstrap draws: 3000; full-run output: results/")
subprocess.run([sys.executable, str(PROJECT / "run_study.py"), "--verify-inputs"],
               cwd=PROJECT, check=True, text=True, capture_output=True)
print("Input integrity passed.")

## Rebuild the study

The full mode repeats training cross-validation, expanding predictions, constrained factor allocation, inference and sensitivity calculations. Progress appears after each stage. The detailed log is stored in `results/notebook_execution.log`; output folders are created automatically.

In [ ]:
def execute_study(options):
    log_path = PROJECT / "results/notebook_execution.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, "-u", str(PROJECT / "run_study.py"), *options]
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        with log_path.open("a", encoding="utf-8") as log:
            for line in process.stdout:
                log.write(line)
                if line.startswith(("Raw source", "[", "Numerical comparison", "Built report", "Study rebuilt", "Stopped:")):
                    print(line.rstrip().replace(str(PROJECT), "<project>"), flush=True)
        status = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    if status:
        tail = "\n".join(log_path.read_text(encoding="utf-8").splitlines()[-25:])
        raise RuntimeError("Study execution failed. Log excerpt:\n" + tail.replace(str(PROJECT), "<project>"))

log = PROJECT / "results/notebook_execution.log"
log.parent.mkdir(parents=True, exist_ok=True)
log.write_text("Full reconstruction\n" if RUN_FULL_ANALYSIS else "Existing-result validation\n", encoding="utf-8")
if RUN_FULL_ANALYSIS:
    execute_study(["--skip-report"])
else:
    for option in ["--validate-only", "--figures-only"]:
        execute_study([option])
print("运行完成，计算结果和图表保存在 results 文件夹。")
if not RUN_FULL_ANALYSIS:
    subprocess.run([sys.executable, str(PROJECT / "tests/test_reproducibility.py")],
                   cwd=PROJECT, check=True)
if not RUN_FULL_ANALYSIS:
    for check in ["test_supplement.py", "test_cross_market.py"]:
        subprocess.run([sys.executable, str(PROJECT / "tests" / check)], cwd=PROJECT, check=True)


## Results

The table uses the current-vintage design, 175-month window, five-factor variance mapping and a 10 bp reallocation charge. Returns are synthetic long-short factor series. The separate momentum cost scenario is reported in the Word document and diagnostic CSVs.

In [ ]:
import pandas as pd
from IPython.display import display, Image

metrics = pd.read_csv(PROJECT / "results/us/metrics.csv")
selected = metrics[(metrics.Design == "Release_corrected_current_vintage") &
                   (metrics.Cohort == "Extended_175") &
                   (metrics.Allocation == "LS5_Variance36_Cap40_Shrink50") &
                   (metrics.Cost_bps == 10) & (metrics.Sample == "Full")].copy()
selected["Annual mean %"] = 100 * selected["Annualized_Mean"]
selected["Annual volatility %"] = 100 * selected["Annualized_Volatility"]
selected["CAGR %"] = 100 * selected["Synthetic_CAGR"]
selected["Max drawdown %"] = 100 * selected["Max_Drawdown"]
columns = ["Strategy", "N_Months", "Annual mean %", "Annual volatility %", "Sharpe", "CAGR %", "Max drawdown %"]
display(selected[columns].reset_index(drop=True).round(3))

### Allocation paths and macro increment

The upper panel shows compounded indexes. The lower panel sums monthly Macro minus historical-mean returns in percentage points; it is an arithmetic difference, not a compounded wealth gap. The dashed line marks the original evaluation window's start.

In [ ]:
display(Image(filename=str(PROJECT / "results/figures/figure1.png"), width=950))

### Five factor weights

Positive weights allocate to synthetic long-short factor return series. The five-factor mapping constrains individual weights to approximately 10%–40%, which keeps strategy return paths close.

In [ ]:
display(Image(filename=str(PROJECT / "results/figures/figure2.png"), width=950))

### Growth and inflation state coverage

Boundaries separate initial training, validation and the original evaluation. Expansion disappears in the original 88-month evaluation under both threshold rules. It still appears in the earlier part of the extended 175-month window.

In [ ]:
display(Image(filename=str(PROJECT / "results/figures/figure3.png"), width=950))

## Additional diagnostics

The first two plots below are supplementary views of mapping dependence and international forecast shrinkage. The cumulative forecast-loss plot also appears in the report, with its distinct benchmarks and sample sizes made explicit.

In [ ]:
display(Image(filename=str(PROJECT / "results/figures/figure4.png"), width=950))

In [ ]:
display(Image(filename=str(PROJECT / "results/figures/figure5.png"), width=950))

The following loss plot uses distinct, explicitly labelled benchmarks and sample sizes. It is raw squared-error accounting, without the Clark–West adjustment. Lines are not calibrated significance tests.

In [ ]:
display(Image(filename=str(PROJECT / "results/figures/figure6.png"), width=950))

### Annual absolute returns and relative gains / 年度绝对收益与相对增量

上图分别复利计算两种组合的年度收益；下图将每月 Macro 减历史均值的收益差按年相加。2022 年贡献约30%的累计算术增量，2024 年贡献更大。两种量的口径不同，不把绝对收益集中解释为相对增益来自同一年。2026年仅含1–7月，不年化。

In [ ]:
display(Image(filename=str(PROJECT / "results/figures/figure7.png"), width=950))

## UK and German allocation checks / 英德配置检验
The frozen US allocation mapping uses the existing three-predictor country Ridge forecasts. JKP themes are conceptual counterparts to US factors, not identical assets; these markets and forecasting designs were previously inspected. The table includes equal weight, which outperforms Macro in the UK. Net differences use each market's matched historical mean and hypothetical 10 bp sleeve charges.

In [ ]:
international = pd.read_csv(PROJECT / "results/cross_market/allocation_summary.csv")
columns = ["Country", "N_Months", "EW_Sharpe", "HM_Sharpe", "Macro_Sharpe", "Macro_vs_HM_Sharpe_Difference", "Macro_vs_HM_CI_Low", "Macro_vs_HM_CI_High"]
display(international[columns].round(6))

In [ ]:
display(Image(filename=str(PROJECT / "results/figures/figure8.png"), width=1050))

UK Macro-minus-mean ΔSharpe is +0.0602 with a six-month-block interval [-0.0699, 0.2644]; Germany is -0.000292 with [-0.00404, 0.00255]. Small German effects and intervals reflect nearly identical paths. UK State-minus-Macro's marginal interval is below zero at blocks three and six but crosses zero at twelve; this differs from the mean-return test and has no Sharpe multiplicity correction.

### Historical IP with common penalties / 固定参数的IP历史版本分解

以下三个设置使用相同配置规则、10bp调仓费用和配对评价月份。固定参数组使用主设计的初始训练惩罚参数，不另选最优参数。均值检验的IP-family Holm只覆盖同一窗口、同一HAC滞后的三个设置，不覆盖完整研究选择。

In [ ]:
ip = pd.read_csv(PROJECT / "results/robustness/ip_design_mean_hac.csv")
ip = ip[(ip.Cohort == "Extended_175") & (ip.HAC_Lags == 3)]
display(ip[["Design_Label", "Annual_Mean_Gain_pp", "p_Two_Sided", "Holm_p_Three_Design_Contrasts"]].reset_index(drop=True).round(4))

## Checks and takeaways / 复现检查

原始数据文件采用严格散列校验；生成数值采用相对容差 `1e-9`、绝对容差 `1e-10`。科学测试覆盖数据时序、组合约束、费用及实现一致性等预先定义的检查。数值复现通过仅说明本规格可重建，不能消除事后规格选择、有限样本功效及执行成本问题。

In [ ]:
validation = json.loads((PROJECT / "results/validation.json").read_text(encoding="utf-8"))
scientific_tests = json.loads((PROJECT / "results/scientific_tests.json").read_text(encoding="utf-8"))
assert validation["all_passed"]
assert scientific_tests["all_passed"]
supplement_tests = json.loads((PROJECT / "results/supplement_tests.json").read_text(encoding="utf-8"))
assert supplement_tests["all_passed"]
cross_market_tests = json.loads((PROJECT / "results/cross_market/semantic_checks.json").read_text(encoding="utf-8"))
assert cross_market_tests["all_passed"]
assert all((PROJECT / f"results/figures/figure{i}.png").is_file() for i in range(1, 9))
print("Numeric reference tables passed:", validation["files"])
print("Relative tolerance:", validation["rtol"], "; absolute tolerance:", validation["atol"])
print("Scientific checks passed:", scientific_tests["tests_run"])
print("Supplement checks passed:", supplement_tests["tests_run"])
print("Cross-market checks passed:", cross_market_tests["tests_run"])
print("计算结果检查通过，8 张图表已生成。")
print("计算结果：", PROJECT / "results")
print("图表目录：", PROJECT / "results/figures")

## Citation / 引用示例

报告文献采用作者–年份方式，例如 **Fama and French (2015)**，完整出处见 Word 的 References。

引用本研究时请使用实际作者与题名；以下是待填写的格式示例，不是一个已发表文献：

> [报告作者] (2026). *Do Macro-Financial Signals Improve US Equity Factor Allocation?*. UTSG 7394 course report, reproducible project version 2026-10-08.

如需引用具体表图，可同时说明评价期、配置名称及 `documentation/report_output_map.csv` 中的结果文件；不要把代码复现成功称为投资策略有效。